In [ ]:
# Video-to-Social Post Generator
"""
Welcome to the Video-to-Social Post Generator!

This beginner-friendly tool helps you turn video URLs into optimized social media posts using AI.
It works with videos from YouTube, TikTok, and Instagram.

Required API Keys:
- Add GEMINI_API_KEY (free at https://makersuite.google.com/app/apikey) or OPENAI_API_KEY to Colab secrets (key icon in sidebar).
- Gemini is default for its creativity and speed.

What It Does:
- Downloads audio from the video URL.
- Transcribes the audio using faster-whisper (with GPU support if available).
- Uses an AI model (Gemini or OpenAI) to generate a platform-optimized post based on your selected style.

Supported Platforms:
- Twitter/X
- LinkedIn
- Instagram
- TikTok
- YouTube (for descriptions)

End-to-End Flow:
1. Choose AI provider (Gemini default).
2. Enter model name or press Enter for default (gemini-1.5-flash).
3. Choose transcription model (large-v3 default for best accuracy).
4. Choose target platform (Twitter/X default).
5. Choose writing style (Professional default).
6. Paste the video URL.
7. Sit back—the tool handles transcription and post generation!
8. Get your ready-to-post content.

Tips:
- Press Enter for defaults to quickly test.
- Runs in a single Colab cell—rerun to try different options.
- Emojis and hashtags are added based on platform defaults.

Let's create some viral content! 🚀
"""

# ============================================================================
# Install Dependencies
# ============================================================================
!pip install -q --upgrade yt-dlp faster-whisper google-generativeai python-dotenv pyyaml openai
!apt-get update -qq && apt-get install -y -qq ffmpeg

# ============================================================================
# Import Libraries
# ============================================================================
import os
import shutil
import tempfile
import yt_dlp
from faster_whisper import WhisperModel
from typing import Dict, Optional
import torch

# Import LLM libraries
from openai import OpenAI
import google.generativeai as genai

# Import Colab secrets
from google.colab import userdata

# ============================================================================
# Platform Definitions
# ============================================================================
PLATFORMS = {
    "1": {
        "name": "Twitter/X",
        "formatting_rules": """
1. Keep the entire post under 280 characters.
2. If the content is too long, structure it as a thread with /1, /2, etc.
3. Use short, punchy sentences.
4. Use line breaks for readability.
""",
        "use_emojis": True,
        "use_hashtags": True,
        "hashtag_count": "2-3",
        "max_tokens": 200
    },
    "2": {
        "name": "LinkedIn",
        "formatting_rules": """
1. Use bold for key phrases with **bold**.
2. Use short paragraphs or bullet points.
3. Professional tone.
""",
        "use_emojis": False,
        "use_hashtags": True,
        "hashtag_count": "3-5",
        "max_tokens": 1000
    },
    "3": {
        "name": "Instagram",
        "formatting_rules": """
1. Use engaging, visual language.
2. Short paragraphs with line breaks.
""",
        "use_emojis": True,
        "use_hashtags": True,
        "hashtag_count": "5-10",
        "max_tokens": 800
    },
    "4": {
        "name": "TikTok",
        "formatting_rules": """
1. Fun, energetic tone.
2. Short and snappy.
""",
        "use_emojis": True,
        "use_hashtags": True,
        "hashtag_count": "4-8",
        "max_tokens": 300
    },
    "5": {
        "name": "YouTube",
        "formatting_rules": """
1. This is for video descriptions.
2. Include timestamps if relevant.
3. Keywords for SEO.
4. Longer content is okay.
""",
        "use_emojis": True,
        "use_hashtags": False,
        "hashtag_count": "",
        "max_tokens": 1500
    }
}

# ============================================================================
# Style Definitions (Persona-Based)
# ============================================================================
STYLES = {
    "1": {
        "name": "Professional",
        "persona": "You are a highly respected professional with a large following on {platform}. You write in a polished, authoritative, and insightful style."
    },
    "2": {
        "name": "Casual",
        "persona": "You are a friendly and relatable influencer on {platform}. You write in a casual, conversational style that feels like chatting with a friend."
    },
    "3": {
        "name": "Storytelling",
        "persona": "You are a captivating storyteller on {platform}. You weave narratives that engage and inspire your audience."
    },
    "4": {
        "name": "Technical",
        "persona": "You are a technical expert on {platform}. You explain complex concepts clearly and precisely."
    },
    "5": {
        "name": "Motivational",
        "persona": "You are an inspiring motivator on {platform}. You use empowering language to encourage and uplift your followers."
    },
    "6": {
        "name": "Question-based",
        "persona": "You are an engagement expert on {platform}. You craft posts that spark discussions through thoughtful questions."
    }
}

# ============================================================================
# VideoTranscriber Class
# ============================================================================
class VideoTranscriber:
    """Downloads video audio and transcribes it using faster-whisper."""

    def __init__(self, model_size: str = "tiny", device: str = "cpu", compute_type: str = "int8"):
        self.model_size = model_size
        self.device = device
        self.compute_type = compute_type
        self.model = None

    def _load_model(self):
        """Lazy load the Whisper model with automatic fallback."""
        if self.model is None:
            print(f"📥 Loading Whisper model: {self.model_size}...")
            try:
                self.model = WhisperModel(
                    self.model_size,
                    device=self.device,
                    compute_type=self.compute_type
                )
                print(f"✅ Model loaded on {self.device}")
            except Exception as e:
                if self.device == "cuda":
                    print(f"⚠️  CUDA error: {str(e)[:100]}")
                    print("🔄 Falling back to CPU...")
                    self.device = "cpu"
                    self.compute_type = "int8"
                    self.model = WhisperModel(
                        self.model_size,
                        device="cpu",
                        compute_type="int8"
                    )
                    print("✅ Model loaded on CPU")
                else:
                    raise

    def _download_audio(self, url: str) -> str:
        """Download audio from video URL with enhanced options to avoid 403 errors."""
        print(f"⬇️  Downloading audio from: {url}")
        temp_dir = tempfile.mkdtemp()
        temp_base = os.path.join(temp_dir, "audio")

        # Enhanced yt-dlp options to bypass 403 errors
        ydl_opts = {
            'format': 'bestaudio/best',
            'outtmpl': temp_base + '.%(ext)s',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }],
            # Enhanced headers to mimic real browser
            'http_headers': {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
                'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
                'Accept-Language': 'en-us,en;q=0.5',
                'Accept-Encoding': 'gzip, deflate',
                'Connection': 'keep-alive',
            },
            # Additional options to avoid blocking
            'extractor_args': {
                'youtube': {
                    'player_client': ['android', 'web'],  # Try Android client first, fallback to web
                }
            },
            'quiet': False,
            'no_warnings': False,
            'retries': 10,  # Retry on failures
            'fragment_retries': 10,
            'ignoreerrors': False,
        }

        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url])

            actual_path = temp_base + '.mp3'
            if os.path.exists(actual_path):
                return actual_path
            else:
                # Try to find any audio file in temp directory
                for file in os.listdir(temp_dir):
                    if file.endswith(('.mp3', '.m4a', '.webm', '.ogg')):
                        return os.path.join(temp_dir, file)
                raise FileNotFoundError(f"Downloaded audio file not found at {actual_path}")
        except Exception as e:
            # Try alternative method with different extractor
            if "403" in str(e) or "Forbidden" in str(e):
                print("⚠️  First attempt failed with 403. Trying alternative method...")
                try:
                    # Alternative options with different player client
                    ydl_opts_alt = {
                        'format': 'bestaudio/best',
                        'outtmpl': temp_base + '.%(ext)s',
                        'postprocessors': [{
                            'key': 'FFmpegExtractAudio',
                            'preferredcodec': 'mp3',
                            'preferredquality': '192',
                        }],
                        'http_headers': {
                            'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
                        },
                        'extractor_args': {
                            'youtube': {
                                'player_client': ['ios', 'android'],  # Try iOS/Android clients
                            }
                        },
                        'quiet': False,
                        'no_warnings': False,
                    }
                    with yt_dlp.YoutubeDL(ydl_opts_alt) as ydl:
                        ydl.download([url])

                    actual_path = temp_base + '.mp3'
                    if os.path.exists(actual_path):
                        return actual_path
                    for file in os.listdir(temp_dir):
                        if file.endswith(('.mp3', '.m4a', '.webm', '.ogg')):
                            return os.path.join(temp_dir, file)
                except Exception as e2:
                    if os.path.exists(temp_dir):
                        shutil.rmtree(temp_dir, ignore_errors=True)
                    raise Exception(f"Failed to download audio after retry: {str(e2)}")

            if os.path.exists(temp_dir):
                shutil.rmtree(temp_dir, ignore_errors=True)
            raise Exception(f"Failed to download audio: {str(e)}")

    def _transcribe_audio(self, audio_path: str) -> Dict:
        """Transcribe audio file using Whisper."""
        self._load_model()
        print("🎤 Transcribing audio...")
        segments, info = self.model.transcribe(
            audio_path,
            beam_size=5,
            language="en",
            task="transcribe"
        )

        transcript_segments = []
        for segment in segments:
            transcript_segments.append({
                'start': segment.start,
                'end': segment.end,
                'text': segment.text.strip()
            })

        return {
            'transcript': transcript_segments,
            'language': info.language,
            'language_probability': info.language_probability
        }

    def transcribe(self, url: str) -> Dict:
        """Main method: Download video and transcribe it."""
        audio_path = None
        temp_dir = None
        try:
            audio_path = self._download_audio(url)
            temp_dir = os.path.dirname(audio_path)
            result = self._transcribe_audio(audio_path)
            return result
        finally:
            if temp_dir and os.path.exists(temp_dir):
                try:
                    shutil.rmtree(temp_dir, ignore_errors=True)
                except:
                    pass

    def get_full_text(self, result: Dict) -> str:
        """Combine all transcript segments into a single text block."""
        return " ".join([seg['text'] for seg in result['transcript']])

# ============================================================================
# SocialPostGenerator Class
# ============================================================================
class SocialPostGenerator:
    """Generates optimized social posts from video transcripts using LLMs."""

    def __init__(self, provider: str = "gemini", model_name: str = "gemini-1.5-flash", api_key: Optional[str] = None):
        self.provider = provider
        self.model_name = model_name

        if provider == "openai":
            api_key = api_key or userdata.get('OPENAI_API_KEY')
            if not api_key:
                raise ValueError("OPENAI_API_KEY not found in Colab secrets. Add it via the key icon in sidebar.")
            self.client = OpenAI(api_key=api_key)
            self.model = None

        elif provider == "gemini":
            api_key = api_key or userdata.get('GEMINI_API_KEY')
            if not api_key:
                raise ValueError("GEMINI_API_KEY not found in Colab secrets. Add it via the key icon in sidebar.")
            genai.configure(api_key=api_key)
            self.model = genai.GenerativeModel(model_name)
            self.client = None

        else:
            raise ValueError(f"Unknown provider: {provider}. Use 'openai' or 'gemini'")

    def generate_post(
        self,
        transcript: str,
        platform: Dict,
        style: Dict,
        temperature: float = 0.7
    ) -> str:
        """Sends transcript to LLM and returns a formatted social post."""
        print(f"🧠 Generating post using {self.provider} ({self.model_name})...")
        print(f"   Platform: {platform['name']}, Style: {style['name']}")

        platform_name = platform["name"]
        persona = style["persona"].format(platform=platform_name)
        formatting_rules = platform["formatting_rules"]
        use_emojis = platform["use_emojis"]
        use_hashtags = platform["use_hashtags"]
        hashtag_count = platform["hashtag_count"]
        max_tokens = platform["max_tokens"]

        system_prompt = f"""{persona}

Your goal is to take a raw video transcript and turn it into a viral post for {platform_name}.

Follow these strict formatting rules:
{formatting_rules}

Start with a strong hook.
Use clean formatting.
End with a call to action.
Remove "umms", "ahhs", and rambling from the transcript. Keep it authentic but polished.
"""

        if use_emojis:
            system_prompt += "\nUse emojis sparingly and appropriately to enhance engagement."
        if use_hashtags:
            system_prompt += f"\nInclude {hashtag_count} relevant hashtags at the end."

        system_prompt += "\n\nOutput only the post content, no meta-commentary."

        user_prompt = f"""Here is the transcript of a video. Write a post based on this:

{transcript}"""

        try:
            if self.provider == "gemini":
                full_prompt = f"{system_prompt}\n\n{user_prompt}"
                response = self.model.generate_content(
                    full_prompt,
                    generation_config=genai.types.GenerationConfig(
                        temperature=temperature,
                        max_output_tokens=max_tokens,
                    )
                )
                return response.text.strip()
            else:
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=temperature,
                    max_tokens=max_tokens,
                )
                return response.choices[0].message.content.strip()
        except Exception as e:
            if self.provider == "gemini":
                error_msg = f"Gemini API error: {str(e)}\nGet free API key at: https://makersuite.google.com/app/apikey"
            else:
                error_msg = f"OpenAI API error: {str(e)}"
            raise Exception(error_msg)

# ============================================================================
# Interactive Execution Pipeline
# ============================================================================

def ask_choice(prompt, options, default_key, display_map):
    print(f"\n{prompt}")
    for k, v in display_map.items():
        print(f"  {k}: {v}")
    choice = input(f"👉 Enter number (default = {default_key}): ").strip()
    if not choice:
        choice = default_key
    return options[choice]

print("=" * 70)
print("🚀 Video-to-Social Post Generator — Interactive Mode")
print("=" * 70)

# ---------------------------------------------------------------------------
# Device Detection
# ---------------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n💻 Device detected: {device.upper()}")

# ---------------------------------------------------------------------------
# Provider (DEFAULT: Gemini)
# ---------------------------------------------------------------------------
provider_options = {
    "1": "gemini",
    "2": "openai"
}
provider_display = {
    "1": "Gemini (default, creative & fast)",
    "2": "OpenAI"
}
provider = ask_choice(
    "Choose AI provider:",
    provider_options,
    default_key="1",
    display_map=provider_display
)

# ---------------------------------------------------------------------------
# Model (DEFAULT based on provider)
# ---------------------------------------------------------------------------
model_name = input(
    f"\n🤖 Enter model name for {provider.upper()} "
    f"(press Enter for default): "
).strip()

if not model_name:
    if provider == "gemini":
        model_name = "gemini-2.5-flash-lite"
    else:
        model_name = "gpt-4o-mini"

# ---------------------------------------------------------------------------
# Whisper Model (DEFAULT: large-v3)
# ---------------------------------------------------------------------------
whisper_options = {
    "1": "tiny",
    "2": "base",
    "3": "small",
    "4": "large-v3"
}
whisper_display = {
    "1": "tiny (fastest, lowest accuracy)",
    "2": "base (balanced)",
    "3": "small (better accuracy)",
    "4": "large-v3 (best, GPU recommended)"
}
whisper_model = ask_choice(
    "Choose transcription model:",
    whisper_options,
    default_key="4",
    display_map=whisper_display
)

# ---------------------------------------------------------------------------
# Platform (DEFAULT: Twitter/X)
# ---------------------------------------------------------------------------
platform_display = {
    "1": "Twitter/X",
    "2": "LinkedIn",
    "3": "Instagram",
    "4": "TikTok",
    "5": "YouTube"
}
platform = ask_choice(
    "Choose target platform:",
    PLATFORMS,
    default_key="1",
    display_map=platform_display
)

# ---------------------------------------------------------------------------
# Writing Style (DEFAULT: Professional)
# ---------------------------------------------------------------------------
style_display = {
    "1": "Professional",
    "2": "Casual",
    "3": "Storytelling",
    "4": "Technical",
    "5": "Motivational",
    "6": "Question-based"
}
style = ask_choice(
    "Choose writing style:",
    STYLES,
    default_key="1",
    display_map=style_display
)

# ---------------------------------------------------------------------------
# URL
# ---------------------------------------------------------------------------
url = input("\n🔗 Paste video URL (YouTube / TikTok / Instagram): ").strip()

if not url:
    raise ValueError("No URL provided.")

# ---------------------------------------------------------------------------
# Initialize Engines
# ---------------------------------------------------------------------------
print("\n⚙️ Initializing engines...")

transcriber = VideoTranscriber(
    model_size=whisper_model,
    device=device,
    compute_type="float16" if device == "cuda" else "int8"
)

writer = SocialPostGenerator(
    provider=provider,
    model_name=model_name
)

print("✅ Ready!\n")

# ---------------------------------------------------------------------------
# Step 1: Transcription
# ---------------------------------------------------------------------------
print("=" * 70)
print("🎧 STEP 1: Transcribing video")
print("=" * 70)

result = transcriber.transcribe(url)
full_text = transcriber.get_full_text(result)

print(
    f"\n✅ Transcript complete "
    f"({len(result['transcript'])} segments | "
    f"{len(full_text)} characters)"
)
print(
    f"🌐 Language: {result['language']} "
    f"({result['language_probability']:.2%})"
)

# ---------------------------------------------------------------------------
# Step 2: Post Generation
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("✍️ STEP 2: Generating post")
print("=" * 70)

post = writer.generate_post(
    transcript=full_text,
    platform=platform,
    style=style,
    temperature=0.7
)

# ---------------------------------------------------------------------------
# Output
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("✨ POST READY ✨")
print("=" * 70)
print(post)
print("=" * 70)

print("\n💡 Tip: Rerun the cell to try different options!")